In [1]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pathlib
import math
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_curve, auc
from tensorflow.keras.utils import Sequence
import numpy as np
import random
from tensorflow.keras import layers
from tensorflow.keras.utils import Sequence
import numpy as np
from collections import defaultdict

2025-06-03 15:16:28.638037: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748963788.820366      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748963788.879173      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
# Setup
batch_size = 32
img_height = 224
img_width = 224
data_dir = pathlib.Path('/kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Augmented Images/Augmented Images/FOLDS_AUG/fold1_AUG/Train')

# Dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size)

class_names = train_ds.class_names
num_classes = len(class_names)

AUTOTUNE = tf.data.AUTOTUNE
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y)).cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y)).cache().prefetch(buffer_size=AUTOTUNE)
# In your dataset pipeline
# ds = ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y))


Found 7518 files belonging to 6 classes.
Using 6015 files for training.


I0000 00:00:1748963811.738522      35 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 7518 files belonging to 6 classes.
Using 1503 files for validation.


In [3]:
class L2Normalize(tf.keras.layers.Layer):
    def call(self, inputs):
        return tf.math.l2_normalize(inputs, axis=1)

In [4]:
# def build_resnet50_encoder(input_shape=(224, 224, 3)):
#     base_model = keras.applications.ResNet50(
#         weights='imagenet',
#         input_shape=input_shape,
#         include_top=False
#     )
#     base_model.trainable = False

#     inputs = keras.Input(shape=input_shape)
#     x = base_model(inputs, training=False)
#     x = keras.layers.GlobalAveragePooling2D()(x)
#     x = keras.layers.Dense(256, activation='relu')(x)
#     x = keras.layers.Dense(128)(x)  # Embedding layer
#     x = L2Normalize()(x)  # ⬅️ Fix applied here
#     return keras.Model(inputs, x, name="ResNet50_Encoder")

In [5]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50

def build_siamese_model(input_shape=(224, 224, 3), trainable=False):
    # Shared ResNet50 Backbone
    def build_base_network(input_shape):
        base_cnn = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
        base_cnn.trainable = trainable

        inputs = tf.keras.Input(shape=input_shape)
        x = layers.Rescaling(1./255)(inputs)  # Normalize input
        x = base_cnn(x)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Lambda(lambda t: tf.math.l2_normalize(t, axis=1))(x)  # L2 normalize
        return Model(inputs, x, name="resnet50_base")

    # Inputs
    input_1 = tf.keras.Input(shape=input_shape)
    input_2 = tf.keras.Input(shape=input_shape)

    # Shared base
    base_network = build_base_network(input_shape)
    embed_1 = base_network(input_1)
    embed_2 = base_network(input_2)

    # L2 distance
    l2_distance = layers.Lambda(lambda tensors: tf.norm(tensors[0] - tensors[1], axis=1, keepdims=True))([embed_1, embed_2])

    # Siamese model
    model = Model(inputs=[input_1, input_2], outputs=l2_distance, name="Siamese_ResNet50")
    return model

In [6]:
def contrastive_accuracy(threshold=0.5):
    def accuracy(y_true, y_pred):
        return tf.reduce_mean(tf.cast(tf.equal(tf.cast(y_true, tf.float32),
                                               tf.cast(y_pred < threshold, tf.float32)), tf.float32))
    return accuracy
    
def contrastive_loss(margin=1.0):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)  # avoid 0 or very large values
        square_pred = tf.square(y_pred)
        margin_square = tf.square(tf.maximum(margin - y_pred, 0))
        return tf.reduce_mean(y_true * square_pred + (1 - y_true) * margin_square)
    return loss


In [7]:
class SiameseDataGenerator(Sequence):
    def __init__(self, dataset, batch_size, **kwargs):
        super().__init__(**kwargs)
        self.dataset = list(dataset)
        self.batch_size = batch_size
        self.image_pairs, self.labels = self._make_pairs()

    def _make_pairs(self):
        label_to_images = defaultdict(list)

        # Collect images per label
        for batch_images, batch_labels in self.dataset:
            for img, lbl in zip(batch_images.numpy(), batch_labels.numpy()):
                label_to_images[lbl].append(img)

        pairs = []
        labels = []

        # Positive pairs (same class)
        for label, images in label_to_images.items():
            n = len(images)
            for i in range(n - 1):
                pairs.append((images[i], images[i + 1]))
                labels.append(1.0)

        # Negative pairs (different classes)
        all_labels = list(label_to_images.keys())
        num_pos = len(pairs)
        for _ in range(num_pos):
            lbl1, lbl2 = np.random.choice(all_labels, 2, replace=False)
            img1 = random.choice(label_to_images[lbl1])
            img2 = random.choice(label_to_images[lbl2])
            pairs.append((img1, img2))
            labels.append(0.0)

        return pairs, labels

    def __len__(self):
        return int(np.ceil(len(self.image_pairs) / self.batch_size))

    def __getitem__(self, index):
        batch_pairs = self.image_pairs[index * self.batch_size:(index + 1) * self.batch_size]
        batch_labels = self.labels[index * self.batch_size:(index + 1) * self.batch_size]

        if len(batch_pairs) == 0:
           return (np.array([]), np.array([])), np.array([])

        img1 = np.stack([pair[0] for pair in batch_pairs])
        img2 = np.stack([pair[1] for pair in batch_pairs])
        labels = np.array(batch_labels, dtype=np.float32)

        return (img1, img2), labels

In [8]:
# y_true: 1 = similar (positive pair), 0 = dissimilar (negative pair)
siamese_model = build_siamese_model(input_shape=(224, 224, 3))
siamese_model.compile(optimizer='adam',
                      loss=contrastive_loss(margin=1.0),
                      metrics=['accuracy'])

train_gen = SiameseDataGenerator(train_ds, batch_size=batch_size)
val_gen = SiameseDataGenerator(val_ds, batch_size=batch_size)

history = siamese_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10


I0000 00:00:1748963857.103951      99 service.cc:148] XLA service 0x7c6d38002ea0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1748963857.104992      99 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1748963859.872366      99 cuda_dnn.cc:529] Loaded cuDNN version 90300


  2/376 ━━━━━━━━━━━━━━━━━━━━ 35s 95ms/step - accuracy: 0.7500 - loss: 0.7499   

I0000 00:00:1748963863.626379      99 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


376/376 ━━━━━━━━━━━━━━━━━━━━ 78s 146ms/step - accuracy: 0.5015 - loss: 0.5014 - val_accuracy: 0.5000 - val_loss: 0.5000
Epoch 2/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 41s 108ms/step - accuracy: 0.5132 - loss: 0.5132 - val_accuracy: 0.5000 - val_loss: 0.5000
Epoch 3/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 41s 109ms/step - accuracy: 0.4368 - loss: 0.4368 - val_accuracy: 0.5000 - val_loss: 0.5000
Epoch 4/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 41s 109ms/step - accuracy: 0.4938 - loss: 0.4938 - val_accuracy: 0.5000 - val_loss: 0.5000
Epoch 5/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 41s 109ms/step - accuracy: 0.4952 - loss: 0.4952 - val_accuracy: 0.5000 - val_loss: 0.5000
Epoch 6/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 41s 109ms/step - accuracy: 0.4617 - loss: 0.4617 - val_accuracy: 0.5000 - val_loss: 0.5000
Epoch 7/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 41s 109ms/step - accuracy: 0.5418 - loss: 0.5418 - val_accuracy: 0.5000 - val_loss: 0.5000
Epoch 8/10
376/376 ━━━━━━━━━━━━━━━━━━━━ 41s 109ms/step - accuracy: 0.4327 - loss: 0.4327 - val

In [ ]:
# Get predictions and true labels
y_true = []
y_pred = []

for x_batch, y_batch in val_gen:
    if len(y_batch) == 0:
        continue  # skip empty batches

    preds = siamese_model.predict(x_batch)
    y_true.extend(y_batch)
    y_pred.extend(preds.squeeze())

y_pred_bin = np.array(y_pred) > 0.5
y_true = np.array(y_true)

# F1 Score
f1 = f1_score(y_true, y_pred_bin, average='macro')
print("Macro F1 Score:", f1)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_bin)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Different", "Same"], yticklabels=["Different", "Same"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_pred)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label='ROC Curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 